# ROS03 · ROS 2 quality of service

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/03_ros2/ROS03_ros_2_quality_of_service.ipynb)

**Track:** ROS 2 · **Skill axis (sheet):** Real-time and Linux · **Time:** about 55 min

**Skill Hub concepts:** ROS 2 quality of service

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

**Languages:** each node task has an optional C++ (rclcpp) cell after it. The grader builds it with colcon and runs the same checker against it.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("ROS03")

## Map: QoS policies and the request/offer rule

A publisher **offers** a QoS, a subscription **requests** one; DDS connects them only if the offer is at least as
strong as the request. An incompatible pair silently exchanges nothing (`ros2 topic info -v` shows why).

| Policy | Values | Compatible when |
|---|---|---|
| Reliability | RELIABLE (retransmit) / BEST_EFFORT (drop) | not (publisher BEST_EFFORT and subscriber RELIABLE) |
| Durability | VOLATILE / TRANSIENT_LOCAL (keep last N for late joiners) | not (publisher VOLATILE and subscriber TRANSIENT_LOCAL) |
| Deadline | max period between messages | publisher's offered period <= subscriber's requested period |
| Liveliness | AUTOMATIC / MANUAL_BY_TOPIC, lease duration | similar offer >= request rule |
| History | KEEP_LAST depth N / KEEP_ALL | local queue only, never incompatible |

| Profile | Reliability | Durability | Depth | Typical topic |
|---|---|---|---|---|
| `qos_profile_sensor_data` | BEST_EFFORT | VOLATILE | 5 | camera, lidar, IMU |
| default (`10`) | RELIABLE | VOLATILE | 10 | commands, state |
| "latched" | RELIABLE | TRANSIENT_LOCAL | 1 | `/robot_description`, maps, static config |

In [ ]:
from roboprep import ros

## Task 1: the compatibility rule in plain Python
QoS dicts have keys `reliability` (`"reliable"`/`"best_effort"`), `durability` (`"volatile"`/`"transient_local"`)
and `deadline` (seconds, or `None` for infinite). Return `(ok, [names of incompatible policies])`.

In [ ]:
def compatible(pub, sub):
    raise NotImplementedError  # TODO

lab.check("rules", compatible)

## Task 2: a latched publisher
Node `map_server_lite` publishes `std_msgs/String` `"map v1"` **once**, at startup, on `/map_lite`, so that a
subscriber that joins seconds later still gets it.

In [ ]:
%%writefile map_publisher.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from rclpy.qos import DurabilityPolicy, QoSProfile, ReliabilityPolicy
from std_msgs.msg import String


class MapLite(Node):
    def __init__(self):
        super().__init__("map_server_lite")
        self.pub = self.create_publisher(String, "/map_lite", 10)   # TODO: QoS for late joiners
        self.pub.publish(String(data="map v1"))


def main():
    rclpy.init()
    node = MapLite()
    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

## Task 3: subscribe to a best-effort sensor
The checker publishes `std_msgs/Float32` on `/scan_fake` with BEST_EFFORT. Node `scan_counter` counts what it
receives and publishes the running count (`std_msgs/Int32`) on `/scan_count` every 0.5 s.

In [ ]:
%%writefile scan_counter.py
import rclpy
from rclpy.executors import ExternalShutdownException
from rclpy.node import Node
from rclpy.qos import qos_profile_sensor_data
from std_msgs.msg import Float32, Int32


class ScanCounter(Node):
    def __init__(self):
        super().__init__("scan_counter")
        self.count = 0
        self.pub = self.create_publisher(Int32, "/scan_count", 10)
        self.create_subscription(Float32, "/scan_fake", self.on_scan, 10)   # TODO: why does this get nothing?
        self.create_timer(0.5, lambda: self.pub.publish(Int32(data=self.count)))

    def on_scan(self, msg):
        self.count += 1


def main():
    rclpy.init()
    node = ScanCounter()
    try:
        rclpy.spin(node)
    except (KeyboardInterrupt, ExternalShutdownException):
        pass


if __name__ == "__main__":
    main()

In [ ]:
lab.check("latched")
lab.check("sensor_qos")

In [ ]:
%%writefile map_publisher.cpp
// C++ (rclcpp) version of Task 2 (optional, same checker). Grade it with lab.check("latched", "map_publisher.cpp"):
// the grader builds it with colcon as package rdl_map_publisher, executable map_publisher (depends on rclcpp, std_msgs).
#include <rclcpp/rclcpp.hpp>
#include <std_msgs/msg/string.hpp>

class MapLite : public rclcpp::Node {
public:
    MapLite() : Node("map_server_lite") {
        pub_ = create_publisher<std_msgs::msg::String>("/map_lite", 10);   // TODO: QoS for late joiners
        std_msgs::msg::String m;
        m.data = "map v1";
        pub_->publish(m);
    }
private:
    rclcpp::Publisher<std_msgs::msg::String>::SharedPtr pub_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<MapLite>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("latched", "map_publisher.cpp")  # uncomment after writing the C++ cell above

In [ ]:
%%writefile scan_counter.cpp
// C++ (rclcpp) version of Task 3 (optional, same checker). Grade it with lab.check("sensor_qos", "scan_counter.cpp"):
// the grader builds it with colcon as package rdl_scan_counter, executable scan_counter (depends on rclcpp, std_msgs).
#include <chrono>
#include <rclcpp/rclcpp.hpp>
#include <std_msgs/msg/float32.hpp>
#include <std_msgs/msg/int32.hpp>
using namespace std::chrono_literals;

class ScanCounter : public rclcpp::Node {
public:
    ScanCounter() : Node("scan_counter") {
        pub_ = create_publisher<std_msgs::msg::Int32>("/scan_count", 10);
        sub_ = create_subscription<std_msgs::msg::Float32>("/scan_fake", 10,   // TODO: why does this get nothing?
            [this](const std_msgs::msg::Float32 &) { ++count_; });
        timer_ = create_wall_timer(500ms, [this] { std_msgs::msg::Int32 m; m.data = count_; pub_->publish(m); });
    }
private:
    int count_ = 0;
    rclcpp::Publisher<std_msgs::msg::Int32>::SharedPtr pub_;
    rclcpp::Subscription<std_msgs::msg::Float32>::SharedPtr sub_;
    rclcpp::TimerBase::SharedPtr timer_;
};

int main(int argc, char **argv) {
    rclcpp::init(argc, argv);
    rclcpp::spin(std::make_shared<ScanCounter>());
    rclcpp::shutdown();
}

In [ ]:
# lab.check("sensor_qos", "scan_counter.cpp")  # uncomment after writing the C++ cell above

See the offered and requested QoS side by side:

In [ ]:
import time
ros.start("python map_publisher.py")
time.sleep(2)
ros.ros("ros2 topic info -v /map_lite | grep -E 'Node name|Endpoint type|Reliability|Durability'")
ros.ros("ros2 topic echo --once --qos-durability transient_local --qos-reliability reliable /map_lite", timeout=15)
ros.stop_all()

## Checkpoint: Math
**M1 (Apply).** KEEP_LAST depth 10 on a 100 Hz topic. Milliseconds of history held?

**M2 (Analyze).** A publisher offers a 20 ms deadline (so it matches a subscriber requesting 25 ms) but really publishes
at 30 Hz. Roughly how many deadline-missed events per second does the subscriber report?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Analyze).** Your node subscribes to a camera driver's images with the default QoS and receives nothing, while
`ros2 topic hz` works. Why?
- a) the image is too large
- b) the driver publishes BEST_EFFORT and your default subscription requests RELIABLE: incompatible, so no data
- c) the domain ID differs

**K2 (Understand).** Explain VOLATILE vs TRANSIENT_LOCAL with `/robot_description` as the example, and what depth
means for each.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `ros03.h`).
3. In `roboprep/checks/ros03.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()